# CLIPSeg rd64-refined — DIMER E2E text-prompted segmentation adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/clipseg-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/clipseg-segmentation-pipeline/blob/main/tutorials/clipseg_segmentation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-CIDAS%2Fclipseg--rd64--refined-ffcc4d?style=flat)](https://huggingface.co/CIDAS/clipseg-rd64-refined) [![Upstream](https://img.shields.io/badge/Upstream-timojl%2Fclipseg-181717?style=flat&logo=github&logoColor=white)](https://github.com/timojl/clipseg) [![arXiv](https://img.shields.io/badge/arXiv-2112.10003-b31b1b.svg)](https://arxiv.org/abs/2112.10003)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot (text-prompted) image segmentation — one image plus 1–16 free-text phrases → one binary mask and probability map per phrase — and bounded supervised fine-tuning of the CLIPSeg decoder on labelled (image, phrase, mask) records, using the pinned `CIDAS/clipseg-rd64-refined` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/clipseg_segmentation_pipeline/`, at revision `085badd6c3f8`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `999e0328d9e10b484360c477313983f9afdd7050` (~605 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `CIDAS/clipseg-rd64-refined` snapshot (a 603 MB `model.safetensors`; no pickle is opened anywhere), fetches the first eight row groups of the FoodSeg103 validation shard from the Hugging Face Hub at an immutable revision with HTTPS range requests (about 43 MB; each row group refused on any SHA-256 or byte-total mismatch), turns each of the 800 images into one (image, phrase, mask) record and splits them by image into 600 / 60 / 140, segments a synthetic scene of drawn shapes through the inference contract with an input manifest and a rejection probe, scores the frozen model over the 140 held-out records (mean IoU, micro IoU, Dice, pixel precision and recall at the threshold) beside an empty-mask and a full-mask baseline, runs a bounded fine-tuning of the CLIPSeg decoder on cached CLIP activations with validation-mIoU epoch selection, scores the held-out records again, re-runs six held-out records and the drawn scene with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify mask parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On a Tesla T4 the default path took about @P:T4_TOTAL_MIN@ minutes of cell time (eight epochs @P:T4_ADAPT_S@ s, frozen scoring of 140 records @P:T4_FROZEN_S@ s); a CUDA runtime is used automatically when present, and the path is practical on CPU too (the build venv cached the 660 training and validation records in about two to three minutes and trained the decoder in seconds per epoch).

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip of images and mask images plus a `masks.csv` (`file`, `mask`, `prompt`, optional `id`; one row per image, at least eight images; a mask image's non-zero pixels are the mask). The records pass through the same validation, image-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the FoodSeg103 sample. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

CLIPSeg is a frozen CLIP ViT-B/16 image encoder and CLIP text encoder joined by a small transformer decoder: the decoder reads the image tower's activations at layers 3, 6 and 9, conditions them on the phrase's CLIP embedding through FiLM, and produces one 352 × 352 logit map per phrase; the carried module passes the logits through a sigmoid, resamples the probability map to the input size and thresholds it into a mask under a caller-owned `threshold` (150,747,746 parameters in all, published under the **Apache-2.0** licence). The output is **an uncalibrated per-pixel sigmoid**: the same map can be a tight or a generous mask depending on the threshold, and **the threshold is a caller-owned request parameter**.

What this notebook adds to inference is **adaptation of the decoder on labelled (image, phrase, mask) records**. The records are food photographs from FoodSeg103, each paired with the name of the ingredient that covers the most pixels (`bread`, `chicken duck`, `steak`, `pie`, …) and that ingredient's pixel mask — a phrase vocabulary and a boundary convention far from the PhraseCut phrases the decoder was trained on, and on them the frozen model already finds the right region roughly: a mean IoU of **@P:FROZEN_MIOU@** on the 140 held-out records in the build record (the full-mask baseline scores @P:FULL_MIOU@). So the honest question is narrow: does a bounded fine-tuning of the 1,127,009-parameter decoder on 600 records — the CLIP towers frozen, exactly as the upstream authors trained it — move the held-out **mean IoU**, **Dice**, **pixel precision** and **pixel recall** on an image-disjoint test split past the frozen model and two **non-adapted baselines**, and what does it do to the drawn shapes the same decoder segments? Nothing here is a claim about your images or your phrases: it is one seeded split of one small labelled set.

**Snapshot note:** the pinned revision ships `model.safetensors` (an 8-file manifest with the tokenizer and processor files) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the processor or the model is constructed. The pipeline runs in **float32 on every device**: the adapter is trained in float32 and overlays without a cast, and CPU, Tesla-class and consumer GPUs then run the same arithmetic.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned labelled mask set, turn it into (image, phrase, mask) records, validate it and split it by image without leakage; segment a drawn scene through the public API and read the output contract correctly (an uncalibrated sigmoid, a caller-owned threshold, a `sample-sanity` report only against masks you drew yourself); measure the frozen model's held-out mean IoU, Dice, pixel precision and recall beside two non-adapted baselines; run a bounded fine-tuning of the decoder with the per-pixel binary cross-entropy, explicit hyperparameters and validation-based epoch selection; evaluate on an image-disjoint test split; look at the adapted masks next to the frozen ones and the references, and at what the drawn scene does after the shared decoder was tuned; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** instance or panoptic segmentation (one binary mask per phrase; masks of different phrases are independent), phrase vocabularies beyond one phrase per record in the adaptation sample, threshold tuning (the threshold is fixed at `MASK_THRESHOLD` for every measurement here), fine-tuning of the CLIP image or text towers, evaluation on PhraseCut or a segmentation benchmark proper (only one seeded 800-record sample is scored here), and any claim that ingredient masks stand in for your images. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Kaggle, Python 3.12; CPU or CUDA). The default path uses CUDA automatically when present. The CLIP towers run `EVAL_BATCH_SIZE` records per forward and the build record measured @P:T4_FROZEN_S@ s to score 140 records and @P:T4_ADAPT_S@ s for the eight epochs (caching the tower activations for 600 + 60 records took @P:T4_CACHE_S@ s) on a Tesla T4, about @P:T4_TOTAL_MIN@ minutes of cell time for the whole path including the pinned install and the downloads. The pinned `torch==2.14.0` install and the 603 MB checkpoint are the large downloads of the run; the row groups are about 43 MB. The activation cache holds about 1.3 GB of half-precision tensors on the host for 600 records.
- **Knowledge:** basic Python, NumPy and PIL; what a per-pixel sigmoid is and why thresholding it is a decision the caller owns; what intersection-over-union and Dice measure and why 140 records from one draw give no dispersion; why a self-drawn scene is a plumbing check while a held-out split of one labelled set is a measurement of that set only.
- **Data contract:** records are `{id, image, prompt, mask}` — `image` a PIL image (or a file decodable by Pillow) with sides within 16..4,096 px, `prompt` one phrase of at most 64 characters (normalised like a query), `mask` a boolean height × width array or a mask image whose non-zero pixels are the mask, with at least one true pixel. Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 8..5,000 records; splitting de-duplicates by decoded pixels so no image lands in two splits. BYOD accepts one zip (or directory) of images and mask images plus a `masks.csv` in the layout named above.
- **Validation is structural, not semantic:** every image and mask is decoded and every phrase checked, but nothing checks that a mask outlines what its phrase names — a mislabelled set is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads eight row groups of `default/validation/0000.parquet` from `https://huggingface.co/datasets/EduardoPacheco/FoodSeg103/resolve/<revision>/` at the immutable parquet-conversion revision `176acc3e…` with HTTPS range requests (the parquet footer plus about 43 MB of row-group bytes out of a 115 MB shard), each row group pinned by SHA-256 and byte total in the carried `samples.py` and refused on any mismatch. FoodSeg103 is published under the Apache-2.0 licence (LARC-CMU-SMU; Wu et al. 2021); nothing is redistributed by this repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `CIDAS/clipseg-rd64-refined` snapshot (~605 MB in total) at revision `999e0328d9e1…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `PIL` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'clipseg-segmentation-pipeline',
    'repository_revision': '085badd6c3f8febb9beedba72958bab31ebc5b25',
    'embedded_module': 'src/clipseg_segmentation_pipeline/pipeline.py',
    'embedded_modules': ['src/clipseg_segmentation_pipeline/metrics.py', 'src/clipseg_segmentation_pipeline/pipeline.py', 'src/clipseg_segmentation_pipeline/samples.py'],
    'module_sha256': '81153d120f741754fabb3624d7c6799e79d46bc818809f28e8df75c4334d0c6a',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/clipseg_segmentation_pipeline/` @ `085badd6c3f8`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/clipseg_segmentation_pipeline/metrics.py`

In [ ]:
"""Corpus-level measures for text-prompted binary segmentation: per-record IoU and Dice at a stated threshold,
their means (macro), the pixel-pooled IoU (micro), pixel precision and recall, and two non-adapted baselines
(empty mask, full-image mask).

Every rate is computed over the supplied records at the supplied threshold; the sigmoid is uncalibrated and no
dispersion is estimated (one seeded split of one sample gives one number)."""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np

METRIC_DEFINITIONS = {
    "miou": "mean over records of the intersection-over-union between the thresholded mask and the reference mask (macro IoU; an empty prediction against a non-empty reference scores 0)",
    "iou_micro": "total intersection pixels over total union pixels across all records (micro IoU; large masks weigh more)",
    "dice": "mean over records of 2·|A∩B| / (|A| + |B|)",
    "pixel_precision": "total true-positive pixels over total predicted pixels",
    "pixel_recall": "total true-positive pixels over total reference pixels",
    "baselines": "empty = no pixel predicted (IoU 0 by construction); full = every pixel predicted (IoU = the reference's area fraction)",
}


def mask_metrics(prediction: np.ndarray, reference: np.ndarray) -> dict[str, float]:
    """IoU, Dice, precision and recall of one boolean prediction against one boolean reference."""
    a = np.asarray(prediction, dtype=bool)
    b = np.asarray(reference, dtype=bool)
    if a.shape != b.shape:
        raise ValueError(f"prediction shape {a.shape} != reference shape {b.shape}")
    inter = int(np.logical_and(a, b).sum())
    union = int(np.logical_or(a, b).sum())
    pred, ref = int(a.sum()), int(b.sum())
    return {
        "iou": inter / union if union else 0.0,
        "dice": 2 * inter / (pred + ref) if (pred + ref) else 0.0,
        "precision": inter / pred if pred else 0.0,
        "recall": inter / ref if ref else 0.0,
        "intersection": inter,
        "union": union,
        "predicted": pred,
        "reference": ref,
    }


def segmentation_metrics(predictions: Sequence[np.ndarray], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Score one boolean mask per record against the record's `mask`."""
    if len(predictions) != len(records):
        raise ValueError(f"{len(predictions)} predictions for {len(records)} records")
    if not records:
        raise ValueError("no records to score")
    rows = []
    for prediction, record in zip(predictions, records, strict=True):
        row = mask_metrics(prediction, record["mask"])
        row["id"] = record["id"]
        row["prompt"] = record["prompt"]
        rows.append(row)
    inter = sum(r["intersection"] for r in rows)
    union = sum(r["union"] for r in rows)
    pred = sum(r["predicted"] for r in rows)
    ref = sum(r["reference"] for r in rows)
    return {
        "n": len(rows),
        "miou": sum(r["iou"] for r in rows) / len(rows),
        "iou_micro": inter / union if union else 0.0,
        "dice": sum(r["dice"] for r in rows) / len(rows),
        "pixel_precision": inter / pred if pred else 0.0,
        "pixel_recall": inter / ref if ref else 0.0,
        "predicted_pixels": pred,
        "reference_pixels": ref,
        "rows": rows,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def empty_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Predict no pixel for every record: IoU, Dice and recall 0 by construction."""
    out = segmentation_metrics([np.zeros(np.asarray(r["mask"]).shape, dtype=bool) for r in records], records)
    out["baseline"] = "empty mask (no pixel predicted)"
    return out


def full_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Predict every pixel for every record: IoU equals the reference's area fraction, recall 1."""
    out = segmentation_metrics([np.ones(np.asarray(r["mask"]).shape, dtype=bool) for r in records], records)
    out["baseline"] = "full-image mask (every pixel predicted)"
    return out

**Module 2/3:** `src/clipseg_segmentation_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Text-prompted (zero-shot) image segmentation with the pinned ``CIDAS/clipseg-rd64-refined`` checkpoint, plus
the adaptation contract for labelled (image, phrase, mask) records: corpus evaluation at a stated threshold,
bounded fine-tuning of the CLIPSeg decoder on cached CLIP activations, and a verified adapter artifact.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the CLIPSeg architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed.
"""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

import hashlib
import json
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "CIDAS/clipseg-rd64-refined"
MODEL_REVISION = "999e0328d9e10b484360c477313983f9afdd7050"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "clipseg-rd64-refined"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Threshold on the per-pixel sigmoid of the decoder logits. 0.5 is the natural cut of a sigmoid and
# the value the smoke run used; the sigmoid is not calibrated, so the deployment owns tuning it on its
# own labelled masks.
MASK_THRESHOLD = 0.5
# Decoder output resolution: logits are 352x352 for every input (preprocessor_config.json resizes to
# 352x352 without preserving aspect ratio); the pipeline resamples the probability map back to the
# input size bilinearly.
LOGIT_SIZE = 352
# Input ceilings. Image cost is bounded by the fixed resize; the side ceiling only guards memory during
# decoding and resampling. Each prompt is one CLIP text query (77-token context); phrases are short.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_PROMPTS = 16
MAX_PROMPT_CHARS = 64
MAX_TEXT_TOKENS = 77
WEIGHTS_FILE = "model.safetensors"

# Adaptation contract: the CLIPSeg decoder (three transformer layers over the CLIP activations extracted at
# `EXTRACT_LAYERS`, the FiLM conditioning on the phrase embedding, the reduce projections and the transposed
# convolution) is the adapter — the part the upstream authors trained; the CLIP vision and text towers stay
# frozen, so their outputs are computed once per record and cached (fp16 on the host).
PARAMETER_COUNT = 150_747_746
DECODER_PARAMETERS = 1_127_009
EXTRACT_LAYERS = (3, 6, 9)
_TRAINABLE_PREFIXES = ("decoder.",)
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = 1
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 5_000
EVAL_BATCH_SIZE = 8
GRAD_CLIP = 1.0


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _weight_digest(root: Path) -> str | None:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        return None
    with open(manifest_path, encoding="utf-8") as handle:
        entries = json.load(handle).get("files", [])
    return next((e["sha256"] for e in entries if e["path"] == WEIGHTS_FILE), None)


def _trainable_names(model: Any) -> list[str]:
    """The decoder's tensors; the CLIP vision and text towers and their projections stay frozen."""
    return [name for name, _ in model.named_parameters() if name.startswith(_TRAINABLE_PREFIXES)]


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    """Refuse an adapter that names another base, another format or a file that does not match its digest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:
        raise ValueError(f"artifact was trained on {base.get('model_id')}@{base.get('revision')}, not {MODEL_ID}@{MODEL_REVISION}")
    if base.get("weight_sha256") != base_sha256:
        raise ValueError("artifact base weight digest does not match the verified snapshot")
    files = manifest.get("files") or []
    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:
        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    size = weights.stat().st_size
    if size != files[0].get("bytes"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get('bytes')}")
    digest = _sha256(weights)
    if digest != files[0].get("sha256"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get('sha256')}")
    names = manifest.get("tensors") or []
    if not names or any(not str(n).startswith(_TRAINABLE_PREFIXES) for n in names):
        raise ValueError("artifact tensors must all belong to the CLIPSeg decoder")
    adapter = manifest.get("adapter") or {}
    threshold = adapter.get("threshold")
    if isinstance(threshold, bool) or not isinstance(threshold, int | float) or not 0.0 <= float(threshold) <= 1.0:
        raise ValueError("artifact manifest must record adapter.threshold, the mask threshold the epoch was selected at")


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection-over-union of two boolean masks of the same shape; the building block for any mIoU."""
    a_bool, b_bool = np.asarray(a, dtype=bool), np.asarray(b, dtype=bool)
    if a_bool.shape != b_bool.shape:
        raise ValueError(f"mask shapes differ: {a_bool.shape} vs {b_bool.shape}")
    union = np.logical_or(a_bool, b_bool).sum()
    return float(np.logical_and(a_bool, b_bool).sum() / union) if union else 0.0


def mask_bbox(mask: np.ndarray) -> list[int] | None:
    """Tight xyxy pixel box around the true pixels of a mask, or ``None`` for an empty mask."""
    rows = np.flatnonzero(np.asarray(mask, dtype=bool).any(axis=1))
    cols = np.flatnonzero(np.asarray(mask, dtype=bool).any(axis=0))
    if rows.size == 0 or cols.size == 0:
        return None
    return [int(cols[0]), int(rows[0]), int(cols[-1]) + 1, int(rows[-1]) + 1]


def format_prompts(prompts: Sequence[str]) -> list[str]:
    """Validate a list of phrases and normalise them: stripped, whitespace-collapsed, lower-cased,
    trailing full stop removed, distinct. The pipeline passes the caller's phrases through otherwise
    unchanged (the upstream examples use plain noun phrases such as "a cat")."""
    if isinstance(prompts, str) or not isinstance(prompts, Sequence):
        raise TypeError("prompts must be a list of phrases, not a single string")
    if not 1 <= len(prompts) <= MAX_PROMPTS:
        raise ValueError(f"prompt count {len(prompts)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")
    cleaned: list[str] = []
    for phrase in prompts:
        if not isinstance(phrase, str):
            raise TypeError(f"prompt must be str, got {type(phrase).__name__}")
        text = " ".join(phrase.split()).strip().rstrip(".").strip().lower()
        if not text:
            raise ValueError("prompt phrases must not be empty")
        if len(text) > MAX_PROMPT_CHARS:
            raise ValueError(
                f"prompt {text[:12]!r}... is {len(text)} chars > MAX_PROMPT_CHARS {MAX_PROMPT_CHARS}"
            )
        cleaned.append(text)
    if len(set(cleaned)) != len(cleaned):
        raise ValueError("prompt phrases must be distinct after normalisation")
    return cleaned


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(name: str, value: Any) -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one PIL.Image.Image (any mode, converted to RGB) plus 1..MAX_PROMPTS free-text phrases",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "prompts": [1, MAX_PROMPTS],
    "prompt_chars": [1, MAX_PROMPT_CHARS],
    "prompt_tokens_per_query": [1, MAX_TEXT_TOKENS],
    "threshold": [0.0, 1.0],
    "preprocessing": (
        f"image converted to RGB and resized to {LOGIT_SIZE}x{LOGIT_SIZE} (aspect ratio not preserved, "
        "ImageNet mean/std); phrases normalised into one CLIP text query each (format_prompts); the "
        f"decoder's {LOGIT_SIZE}x{LOGIT_SIZE} logits are passed through a sigmoid and resampled "
        "bilinearly to the input size; the mask is probability >= threshold"
    ),
    "output": (
        "per prompt: a boolean mask and a float32 probability map at input resolution, the mask's area "
        "fraction, tight box and maximum probability; probabilities are uncalibrated sigmoids"
    ),
}


def _check_inputs(image: Any, prompts: Any, threshold: Any) -> tuple[Image.Image, list[str], float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``segment`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    queries = format_prompts(prompts)
    checked = _check_threshold("threshold", threshold)
    return rgb, queries, checked


def validate_inputs(
    image: Image.Image,
    prompts: Sequence[str],
    *,
    threshold: float = MASK_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``segment`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, queries, checked = _check_inputs(image, prompts, threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (segment takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
                "n_prompts": len(prompts),
            }
        ],
        "queries": queries,
        "threshold": checked,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_masks: Mapping[str, np.ndarray] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference_masks`` (phrase -> boolean mask at input resolution) the report carries one
    ``mask_iou`` entry per reference and their mean (``miou``) as sample-sanity evidence; without them
    the verdict is ``not-measurable`` and the report says what labelled data would make the task
    measurable.
    """
    segments = list(result["segments"])
    by_prompt = {segment["prompt"]: segment for segment in segments}
    base = {
        "task": "zero-shot (text-prompted) binary segmentation, one mask per phrase",
        "decision_rule": (
            "each phrase yields a per-pixel sigmoid over the decoder logits; a pixel belongs to the mask "
            "when that sigmoid reaches the threshold; the sigmoid is uncalibrated and masks of different "
            "phrases are independent (they may overlap or leave pixels unassigned)"
        ),
        "threshold": result.get("threshold", MASK_THRESHOLD),
        "sample_kind": sample_kind,
        "n_prompts": len(segments),
        "area_fractions": {segment["prompt"]: segment["area_fraction"] for segment in segments},
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not reference_masks:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no reference masks were supplied for the evaluated image",
            "needs": (
                "labelled masks on your own images with a phrase vocabulary matching the prompts, scored "
                "per phrase with mask_iou and aggregated into mean IoU at a stated threshold; no such "
                "labelled set ships with this repository"
            ),
        }
    metrics = []
    for phrase, reference in reference_masks.items():
        key = format_prompts([phrase])[0]
        if key not in by_prompt:
            raise ValueError(f"reference phrase {phrase!r} was not among the segmented prompts")
        metrics.append(
            {
                "id": "mask_iou",
                "reference": key,
                "value": mask_iou(by_prompt[key]["mask"], reference),
                "reference_area_fraction": float(np.asarray(reference, dtype=bool).mean()),
                "predicted_area_fraction": by_prompt[key]["area_fraction"],
                "estimation": "one reference mask per phrase on a single scene, no dispersion estimate",
            }
        )
    miou = sum(entry["value"] for entry in metrics) / len(metrics)
    metrics.append(
        {
            "id": "miou",
            "value": miou,
            "estimation": f"mean of {len(metrics)} mask_iou value(s) on one scene, no dispersion estimate",
        }
    )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics) - 1} reference mask(s) on one tutorial sample; geometry sanity evidence, "
            "not a segmentation benchmark"
        ),
        "needs": (
            "a labelled mask set from the deployment domain with a matching phrase vocabulary for any "
            "mean-IoU claim"
        ),
    }


@dataclass
class ClipSegSegmentationPipeline:
    """Text-prompted (zero-shot) binary segmentation over the pinned CLIPSeg rd64-refined checkpoint."""

    _runner: Callable[[Image.Image, list[str]], np.ndarray]
    device: str
    _batch_runner: Callable[[list[Image.Image], list[str]], list[np.ndarray]] | None = None
    _model: Any = None
    _processor: Any = None
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ClipSegSegmentationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import CLIPSegForImageSegmentation, CLIPSegProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = CLIPSegProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = CLIPSegForImageSegmentation.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, dtype=torch.float32, **kwargs
        )
        model = model.to(resolved_device).eval()
        for param in model.parameters():
            param.requires_grad_(False)
        weight_sha256 = _weight_digest(root) if (root / MANIFEST_NAME).is_file() else None

        def runner(image: Image.Image, queries: list[str]) -> np.ndarray:
            # One image copy per phrase: CLIPSeg conditions the decoder on each text query separately.
            inputs = processor(
                text=queries, images=[image] * len(queries), padding=True, return_tensors="pt"
            ).to(resolved_device)
            with torch.inference_mode():
                logits = model(**inputs).logits
            if logits.dim() == 2:  # a single prompt may come back squeezed
                logits = logits.unsqueeze(0)
            probs = torch.sigmoid(logits).unsqueeze(1)
            probs = torch.nn.functional.interpolate(
                probs, size=(image.height, image.width), mode="bilinear", align_corners=False
            )
            return probs.squeeze(1).float().cpu().numpy()

        def batch_runner(images: list[Image.Image], queries: list[str]) -> list[np.ndarray]:
            # One phrase per image: each (image, phrase) pair is one decoder query.
            inputs = processor(text=queries, images=images, padding=True, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                logits = model(**inputs).logits
            if logits.dim() == 2:
                logits = logits.unsqueeze(0)
            probs = torch.sigmoid(logits).unsqueeze(1)
            out = []
            for k, image in enumerate(images):
                resampled = torch.nn.functional.interpolate(probs[k : k + 1], size=(image.height, image.width), mode="bilinear", align_corners=False)
                out.append(resampled[0, 0].float().cpu().numpy())
            return out

        return cls(runner, resolved_device, batch_runner, model, processor, weight_sha256, None)

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained")
        return self._model, self._processor

    def segment(
        self,
        image: Image.Image,
        prompts: Sequence[str],
        *,
        threshold: float = MASK_THRESHOLD,
    ) -> dict[str, Any]:
        """Segment each phrase in ``prompts``; masks and probability maps are at input resolution."""
        rgb, queries, checked = _check_inputs(image, prompts, threshold)
        probs = np.asarray(self._runner(rgb, queries), dtype=np.float32)
        if probs.shape != (len(queries), rgb.height, rgb.width):
            raise RuntimeError(
                f"backend returned probability maps of shape {probs.shape}, "
                f"expected {(len(queries), rgb.height, rgb.width)}"
            )
        if probs.min() < 0.0 or probs.max() > 1.0:
            raise RuntimeError("backend returned probabilities outside [0, 1]")
        segments = []
        for query, prob in zip(queries, probs, strict=True):
            mask = prob >= checked
            segments.append(
                {
                    "prompt": query,
                    "mask": mask,
                    "probability": prob,
                    "area_fraction": float(mask.mean()),
                    "max_probability": float(prob.max()),
                    "bbox": mask_bbox(mask),
                }
            )
        return {
            "segments": segments,
            "queries": queries,
            "threshold": checked,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ------------------------------------------------------------------------------------------------------
    # Adaptation contract: batched (image, phrase) segmentation, corpus evaluation, bounded fine-tuning, artifacts
    # ------------------------------------------------------------------------------------------------------

    def segment_batch(
        self,
        pairs: Sequence[tuple[Image.Image, str]],
        *,
        threshold: float = MASK_THRESHOLD,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> list[dict[str, Any]]:
        """Segment one phrase per image for many (image, phrase) pairs, `batch_size` pairs per forward; one
        ``{prompt, mask, probability, area_fraction, max_probability, bbox}`` per pair, in order. With an injected
        runner and no batch runner the pairs are segmented one by one through the runner."""
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        checked = [_check_inputs(image, [prompt], threshold) for image, prompt in pairs]
        cut = _check_threshold("threshold", threshold)
        out: list[dict[str, Any]] = []
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            if self._batch_runner is not None:
                probs = self._batch_runner([rgb for rgb, _, _ in batch], [q[0] for _, q, _ in batch])
            else:
                probs = [np.asarray(self._runner(rgb, q), dtype=np.float32)[0] for rgb, q, _ in batch]
            if len(probs) != len(batch):
                raise RuntimeError(f"backend returned {len(probs)} probability maps for {len(batch)} pairs")
            for (rgb, queries, _), prob in zip(batch, probs, strict=True):
                prob = np.asarray(prob, dtype=np.float32)
                if prob.shape != (rgb.height, rgb.width):
                    raise RuntimeError(f"backend returned a probability map of shape {prob.shape}, expected {(rgb.height, rgb.width)}")
                if prob.min() < 0.0 or prob.max() > 1.0:
                    raise RuntimeError("backend returned probabilities outside [0, 1]")
                mask = prob >= cut
                out.append({"prompt": queries[0], "mask": mask, "probability": prob, "area_fraction": float(mask.mean()), "max_probability": float(prob.max()), "bbox": mask_bbox(mask)})
            if progress is not None:
                progress(len(out), len(checked))
        return out

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        threshold: float = MASK_THRESHOLD,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> dict[str, Any]:
        """Segment every validated record's phrase on its image and score the thresholded masks against the record
        masks with ``metrics.segmentation_metrics`` (mean IoU, micro IoU, Dice, pixel precision and recall). Works
        with an injected runner too."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import segmentation_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        items = self.segment_batch([(r["image"], r["prompt"]) for r in checked], threshold=threshold, batch_size=batch_size, progress=progress)
        metrics = segmentation_metrics([item["mask"] for item in items], checked)
        metrics.update(
            {
                "threshold": float(threshold),
                "predicted_area_fraction": sum(item["area_fraction"] for item in items) / len(items),
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _encode(self, records: Sequence[Mapping[str, Any]], batch_size: int, progress: Callable[[int, int], None] | None = None) -> tuple[list[Any], Any, Any]:
        """Run the frozen towers once per record: the CLIP vision activations at `EXTRACT_LAYERS` (stored fp16 on
        the host), the phrase embeddings, and the reference masks resampled to the decoder's logit grid."""
        model, processor = self._require_model()
        import torch

        activations: list[list[Any]] = [[] for _ in EXTRACT_LAYERS]
        conditionals, targets = [], []
        with torch.no_grad():
            for start in range(0, len(records), batch_size):
                batch = records[start : start + batch_size]
                inputs = processor(text=[r["prompt"] for r in batch], images=[r["image"] for r in batch], padding=True, return_tensors="pt").to(self.device)
                vision = model.clip.vision_model(pixel_values=inputs["pixel_values"], output_hidden_states=True)
                for slot, layer in enumerate(EXTRACT_LAYERS):
                    activations[slot].append(vision.hidden_states[layer + 1].to("cpu", torch.float16))
                conditionals.append(model.clip.get_text_features(inputs["input_ids"], attention_mask=inputs["attention_mask"]).to("cpu"))
                resized = [torch.nn.functional.interpolate(torch.from_numpy(np.asarray(r["mask"], dtype=np.float32))[None, None], size=(LOGIT_SIZE, LOGIT_SIZE), mode="nearest")[0, 0] for r in batch]
                targets.append(torch.stack(resized))
                if progress is not None:
                    progress(min(start + batch_size, len(records)), len(records))
        return [torch.cat(slot) for slot in activations], torch.cat(conditionals), torch.cat(targets)

    def _decoder_logits(self, activations: Sequence[Any], conditionals: Any) -> Any:
        """The decoder on cached tower outputs: exactly what `CLIPSegForImageSegmentation.forward` computes after
        its frozen CLIP steps (parity with the full forward is asserted by the model-backed tests)."""
        model, _ = self._require_model()
        import torch

        return model.decoder([a.to(self.device, torch.float32) for a in activations], conditionals.to(self.device, torch.float32)).logits

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None,
        *,
        epochs: int = 8,
        lr: float = 3e-4,
        batch_size: int = 8,
        seed: int = 0,
        threshold: float = MASK_THRESHOLD,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the CLIPSeg decoder on labelled (image, phrase, mask) records with the per-pixel
        binary cross-entropy over the decoder's logit grid — the loss the upstream authors trained the decoder
        with. The frozen CLIP towers are run once per record under no gradient and their outputs cached (the
        vision activations at `EXTRACT_LAYERS` and the phrase embedding), so each step runs only the decoder; the
        logits equal the full model's exactly. AdamW (no weight decay), gradient clipping at `GRAD_CLIP`, seeded
        shuffling, no scheduler, no augmentation. Epoch 0 records the frozen model's validation rates at
        `threshold`; the epoch with the highest validation mean IoU (the earliest on ties) is kept. On any
        exception the frozen decoder is restored."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import segmentation_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 100:
            raise ValueError("epochs must be an int in 1..100")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 128:
            raise ValueError("batch_size must be an int in 1..128")
        if not isinstance(lr, int | float) or isinstance(lr, bool) or not 0 < lr <= 1e-2:
            raise ValueError("lr must be a number in (0, 1e-2]")
        cut = _check_threshold("threshold", threshold)
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        model, _processor = self._require_model()
        import torch

        started = time.perf_counter()
        names = _trainable_names(model)
        params = {name: param for name, param in model.named_parameters() if name in set(names)}
        n_trainable = sum(p.numel() for p in params.values())
        backup = {name: param.detach().clone() for name, param in params.items()}
        previous_adapter = self.adapter
        cudnn_flags = torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False
        try:
            model.eval()
            activations, conditionals, targets = self._encode(train_checked, EVAL_BATCH_SIZE)
            cached_val = self._encode(val_checked, EVAL_BATCH_SIZE) if val_checked is not None else None
            cache_seconds = round(time.perf_counter() - started, 3)

            def score_val() -> dict[str, Any] | None:
                if val_checked is None or cached_val is None:
                    return None
                with torch.no_grad():
                    masks = []
                    for start in range(0, len(val_checked), EVAL_BATCH_SIZE):
                        logits = self._decoder_logits([a[start : start + EVAL_BATCH_SIZE].float() for a in cached_val[0]], cached_val[1][start : start + EVAL_BATCH_SIZE])
                        probs = torch.sigmoid(logits).unsqueeze(1)
                        for k, record in enumerate(val_checked[start : start + EVAL_BATCH_SIZE]):
                            prob = torch.nn.functional.interpolate(probs[k : k + 1], size=(record["image"].height, record["image"].width), mode="bilinear", align_corners=False)[0, 0]
                            masks.append((prob >= cut).cpu().numpy())
                m = segmentation_metrics(masks, val_checked)
                return {k: m[k] for k in ("miou", "iou_micro", "dice", "pixel_precision", "pixel_recall", "n")}

            for name, param in model.named_parameters():
                param.requires_grad_(name in params)
            history: list[dict[str, Any]] = [{"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}]
            if progress is not None:
                progress(history[-1])
            best_epoch, best_score = 0, (history[0]["val"] or {}).get("miou", -1.0)
            best_state = {name: param.detach().clone() for name, param in params.items()}
            optimizer = torch.optim.AdamW(list(params.values()), lr=lr, weight_decay=0.0)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            order = list(range(len(train_checked)))
            for epoch in range(1, epochs + 1):
                rng.shuffle(order)
                model.decoder.train()
                total, steps = 0.0, 0
                for start in range(0, len(order), batch_size):
                    idx = torch.tensor(order[start : start + batch_size])
                    optimizer.zero_grad(set_to_none=True)
                    logits = self._decoder_logits([a[idx].float() for a in activations], conditionals[idx])
                    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets[idx].to(self.device))
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(list(params.values()), GRAD_CLIP)
                    optimizer.step()
                    total += float(loss.detach())
                    steps += 1
                model.eval()
                entry = {"epoch": epoch, "train_loss": round(total / max(steps, 1), 5), "val": score_val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                score = (entry["val"] or {}).get("miou")
                if val_checked is None or (score is not None and score > best_score):
                    best_epoch, best_score = epoch, score if score is not None else best_score
                    best_state = {name: param.detach().clone() for name, param in params.items()}
            with torch.no_grad():
                for name, param in params.items():
                    param.copy_(best_state[name])
        except BaseException:
            with torch.no_grad():
                for name, param in params.items():
                    param.copy_(backup[name])
            model.eval()
            self.adapter = previous_adapter
            raise
        finally:
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags
        self.adapter = {
            "threshold": cut,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "extract_layers": list(EXTRACT_LAYERS),
            "epochs": epochs,
            "batch_size": batch_size,
            "best_epoch": best_epoch,
            "selection": "highest validation mean IoU" if val_checked is not None else "final epoch (no validation split)",
            "loss": f"per-pixel binary cross-entropy over the {LOGIT_SIZE}x{LOGIT_SIZE} decoder logits against the reference mask resampled to that grid; computed on cached CLIP activations",
            "lr": float(lr),
            "seed": seed,
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "cache_seconds": cache_seconds,
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained tensors as safetensors plus a manifest naming the base, the digests, the threshold and
        the training configuration. Requires a prior `adapt`."""
        model, _processor = self._require_model()  # refuse before importing torch
        import torch
        from safetensors.torch import save_file

        if self.adapter is None:
            raise RuntimeError("nothing to save: call adapt() first")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHTS_FILE, "weight_sha256": self.weight_sha256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": names,
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact
        tensor set. Refuses tensors outside the decoder."""
        model, _processor = self._require_model()  # refuse before importing safetensors
        from safetensors.torch import load_file

        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        expected = _trainable_names(model)
        if sorted(manifest["tensors"]) != sorted(expected):
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != sorted(expected):
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        for name, tensor in tensors.items():
            if tuple(tensor.shape) != tuple(state[name].shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}
        return dict(self.adapter)

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> ClipSegSegmentationPipeline:
        """Check the adapter manifest against the base snapshot's recorded weight digest, load the verified base, then
        overlay the adapter (checked again, and the tensor set, before deserialising). A refused manifest never loads
        a model."""
        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        _check_artifact_manifest(manifest, artifact, _weight_digest(root) or "")
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 3/3:** `src/clipseg_segmentation_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled (image, phrase, mask) datasets for the adaptation contract: the digest-pinned FoodSeg103 sample, the
record contract and its structural validation, image-disjoint splitting, and the BYOD loader.

A record is ``{id, image, prompt, mask}`` where ``image`` is a PIL image (sides within the pipeline's ceilings),
``prompt`` the phrase the mask answers (normalised like ``format_prompts`` normalises a query) and ``mask`` a
boolean array (or a Pillow-decodable image whose non-zero pixels are the mask) of the image's height × width
with at least one true pixel.

The default sample is drawn from FoodSeg103 (Wu et al. 2021, LARC-CMU-SMU; **Apache-2.0**; a curated sample of
Recipe1M food photographs with pixel-wise ingredient masks) as converted to parquet by the Hugging Face Hub at an
immutable revision: the first ``CORPUS_ROW_GROUPS`` row groups of the validation shard are read with HTTPS range
requests (about 5 MB each; the shard's declared size is checked first and every row group's content is refused
unless its SHA-256 matches the pin). Each image yields one record: the ingredient class that covers the most
pixels becomes the phrase (its FoodSeg103 name, e.g. ``bread``, ``chicken duck``, ``steak``) and that class's
pixels the mask; images whose largest class is ``background`` or ``other ingredients`` are skipped (none of the
800 are). The domain gap to the model's PhraseCut training distribution is the point of the sample: the frozen
model must already know what the phrase means, and the adapter can only sharpen where it draws the boundary.
"""
# ruff: noqa: E501  -- record and pin literals are kept on single lines

from __future__ import annotations

import csv
import hashlib
import io
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MODEL_ID, format_prompts, validate_image` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "FoodSeg103 (validation split), first eight parquet row groups"
CORPUS_REPO = "EduardoPacheco/FoodSeg103"
CORPUS_REVISION = "176acc3edd2432ee126bda6fb01469eadeb018df"  # refs/convert/parquet commit on the Hub
CORPUS_FILE = "default/validation/0000.parquet"
CORPUS_BYTES = 115_186_173
CORPUS_ROWS = 2_135
CORPUS_ROW_GROUPS = 8  # of 22; 100 images each
CORPUS_LICENSE = "Apache-2.0 (FoodSeg103, LARC-CMU-SMU; Wu, Fu, Liu, Lim, Hoi, Sun, ACM MM 2021; images curated from Recipe1M)"
CORPUS_URL = f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/{CORPUS_FILE}"
# SHA-256 over the concatenated image bytes + label-mask bytes + UTF-8 id of each row, in row order, and that byte total.
ROW_GROUP_PINS: dict[int, tuple[str, int]] = {
    0: ("931827f16c33a0548ad26c456c8495fc3949c79f7950869eabaff44f748e8f78", 5_090_593),
    1: ("a784bb0f23e413eef8d0dad90e504f398bde1a44d2110dced06bdea8953f08bf", 5_207_382),
    2: ("f8a8e15d28c6678d344324b3c972008e642878b25ebff9f481196707c39fc32e", 5_467_905),
    3: ("adebbbc59740e87c5eb0578f029174872bd95b7fa5b1178c708e61c766b73fa5", 6_071_430),
    4: ("fd181120d4966d20e47742232e4102d45247b3f14cf7e2a8733b9d08d1aa152d", 4_758_783),
    5: ("c3af4dda9af15abf54d5f4494fd07df925d57c1d9ec654eb30a6c1bd30f50b61", 5_020_081),
    6: ("0597cb610e859eb400c746dfba37abc446afd34d897b573b1c68453c5c9d80d9", 4_784_703),
    7: ("9e9aac90865e09124fa6c3e89cd676ef2dcd65f87532c002574a545671bae4a9", 5_899_238),
}
DEFAULT_CACHE_DIR = Path("weights") / "foodseg103"

# The FoodSeg103 class vocabulary (dataset card, ids 0..103); 0 = background and 103 = other ingredients never become a phrase.
FOODSEG103_CLASSES = ("background", "candy", "egg tart", "french fries", "chocolate", "biscuit", "popcorn", "pudding", "ice cream", "cheese butter", "cake", "wine", "milkshake", "coffee", "juice", "milk", "tea", "almond", "red beans", "cashew", "dried cranberries", "soy", "walnut", "peanut", "egg", "apple", "date", "apricot", "avocado", "banana", "strawberry", "cherry", "blueberry", "raspberry", "mango", "olives", "peach", "lemon", "pear", "fig", "pineapple", "grape", "kiwi", "melon", "orange", "watermelon", "steak", "pork", "chicken duck", "sausage", "fried meat", "lamb", "sauce", "crab", "fish", "shellfish", "shrimp", "soup", "bread", "corn", "hamburg", "pizza", "hanamaki baozi", "wonton dumplings", "pasta", "noodles", "rice", "pie", "tofu", "eggplant", "potato", "garlic", "cauliflower", "tomato", "kelp", "seaweed", "spring onion", "rape", "ginger", "okra", "lettuce", "pumpkin", "cucumber", "white radish", "carrot", "asparagus", "bamboo shoots", "broccoli", "celery stick", "cilantro mint", "snow peas", "cabbage", "bean sprouts", "onion", "pepper", "green beans", "French beans", "king oyster mushroom", "shiitake", "enoki mushroom", "oyster mushroom", "white button mushroom", "salad", "other ingredients")
EXCLUDED_CLASS_IDS = (0, 103)

SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 600, "validation": 60, "test": 140}  # of the 800 images the eight row groups hold
SAMPLE_DIGEST = "d8123852b1f1c8dd7054c77ab36cf408738f48c7902678109d620165c2a6d4bf"  # dataset_digest over the three default splits together; tests pin it
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MIN_MASK_PIXELS = 1
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to read a
    parquet footer and a few row groups without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}", "User-Agent": "clipseg-segmentation-pipeline"})
        with urllib.request.urlopen(request, timeout=300) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _declared_size(url: str) -> int:
    request = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "clipseg-segmentation-pipeline"})
    with urllib.request.urlopen(request, timeout=60) as response:  # noqa: S310 (pinned https URL)
        length = response.headers.get("Content-Length")
    if length is None:
        raise ValueError(f"{url}: no Content-Length in the HEAD response")
    return int(length)


def _group_digest(rows: Sequence[Mapping[str, Any]]) -> tuple[str, int]:
    digest, total = hashlib.sha256(), 0
    for row in rows:
        for chunk in (row["image"]["bytes"], row["label"]["bytes"], str(row["id"]).encode("utf-8")):
            digest.update(chunk)
            total += len(chunk)
    return digest.hexdigest(), total


def fetch_corpus(
    *, cache_dir: str | Path | None = None, groups: Sequence[int] | None = None, opener: Any = None
) -> dict[int, list[dict[str, Any]]]:
    """Return the pinned row groups as lists of `{image, label, id}` (JPEG/PNG bytes, label-mask PNG bytes, source
    id), from the cache (one parquet file per row group) or the Hub (footer + the row groups it needs, over range
    requests). Every row group's decoded content is refused unless its SHA-256 and byte total match
    `ROW_GROUP_PINS`; a fresh fetch also checks the shard's declared size and row count."""
    import pyarrow.parquet as pq

    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    wanted = list(groups) if groups is not None else sorted(ROW_GROUP_PINS)
    out: dict[int, list[dict[str, Any]]] = {}
    reader = None
    for group in wanted:
        if group not in ROW_GROUP_PINS:
            raise ValueError(f"row group {group} has no pin; pinned groups are {sorted(ROW_GROUP_PINS)}")
        local = cache / f"validation-rg{group}.parquet"
        rows: list[dict[str, Any]] | None = None
        if local.is_file():
            rows = pq.read_table(local).to_pylist()
            if _group_digest(rows) != ROW_GROUP_PINS[group]:
                rows = None  # stale or corrupt cache: refetch
        if rows is None:
            if reader is None:
                if opener is not None:
                    reader = pq.ParquetFile(opener(CORPUS_URL))
                else:
                    declared = _declared_size(CORPUS_URL)
                    if declared != CORPUS_BYTES:
                        raise ValueError(f"{CORPUS_FILE}: declared size {declared} != pinned {CORPUS_BYTES}")
                    reader = pq.ParquetFile(io.BufferedReader(_HttpRangeFile(CORPUS_URL, CORPUS_BYTES), buffer_size=1 << 20))
                if reader.metadata.num_rows != CORPUS_ROWS:
                    raise ValueError(f"{CORPUS_FILE}: {reader.metadata.num_rows} rows, pinned {CORPUS_ROWS}")
            table = reader.read_row_group(group, columns=["image", "label", "id"])
            rows = table.to_pylist()
            digest, total = _group_digest(rows)
            if (digest, total) != ROW_GROUP_PINS[group]:
                raise ValueError(f"{CORPUS_FILE} row group {group}: sha256 {digest} / {total} bytes != pinned {ROW_GROUP_PINS[group]}")
            pq.write_table(table, local)
        out[group] = [{"image": r["image"]["bytes"], "label": r["label"]["bytes"], "id": int(r["id"])} for r in rows]
    return out


def largest_class(label: np.ndarray) -> int | None:
    """The FoodSeg103 class id covering the most pixels, background and 'other ingredients' excluded; None if no
    other class is present."""
    ids, counts = np.unique(np.asarray(label), return_counts=True)
    candidates = [(int(c), int(i)) for i, c in zip(ids, counts, strict=True) if int(i) not in EXCLUDED_CLASS_IDS and 0 <= int(i) < len(FOODSEG103_CLASSES)]
    if not candidates:
        return None
    return max(candidates)[1]


def read_corpus(groups: Mapping[int, Sequence[Mapping[str, Any]]]) -> list[dict[str, Any]]:
    """Decode the verified row groups into records: one per image, the largest ingredient class as the phrase and
    its pixels as the mask (images with no ingredient class besides background / other are skipped)."""
    out = []
    for group in sorted(groups):
        for index, row in enumerate(groups[group]):
            label = np.asarray(Image.open(io.BytesIO(row["label"])))
            class_id = largest_class(label)
            if class_id is None:
                continue
            image = Image.open(io.BytesIO(row["image"]))
            image.load()
            out.append(
                {
                    "id": f"foodseg103-val-{group * 100 + index}",
                    "image": image.convert("RGB"),
                    "prompt": FOODSEG103_CLASSES[class_id],
                    "mask": label == class_id,
                    "class_id": class_id,
                    "source_id": row["id"],
                    "source_row_group": group,
                }
            )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None
) -> dict[str, list[dict[str, Any]]]:
    """Seeded image-level draw: shuffle the records and cut `sizes` (train / validation / test) in order."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    pool = [dict(r) for r in records]
    random.Random(seed).shuffle(pool)
    needed = sum(sizes.values())
    if len(pool) < needed:
        raise ValueError(f"only {len(pool)} records available, need {needed}")
    out, cursor = {}, 0
    for name, count in sizes.items():
        out[name] = pool[cursor : cursor + count]
        cursor += count
    return out


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir)), seed=seed)


# ---------------------------------------------------------------------------------------------------------
# Record contract
# ---------------------------------------------------------------------------------------------------------


def _open(image: Any, where: str) -> Image.Image:
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{where}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{where}: must be a PIL.Image.Image or a file path")
    return image


def coerce_mask(mask: Any, size: tuple[int, int], where: str = "mask") -> np.ndarray:
    """A boolean height × width array from a boolean/integer array or a Pillow-decodable image (non-zero = true)."""
    if isinstance(mask, str | Path):
        mask = _open(mask, where)
    if isinstance(mask, Image.Image):
        array = np.asarray(mask.convert("L")) > 0
    else:
        try:
            array = np.asarray(mask)
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"{where}: must be a boolean array or an image") from exc
        if array.dtype != bool:
            if not np.issubdtype(array.dtype, np.number):
                raise ValueError(f"{where}: must be a boolean or numeric array")
            array = array != 0
    width, height = size
    if array.ndim != 2 or array.shape != (height, width):
        raise ValueError(f"{where}: shape {array.shape} does not match the image's height x width {(height, width)}")
    if int(array.sum()) < MIN_MASK_PIXELS:
        raise ValueError(f"{where}: the mask has no true pixel")
    return array


def _check_record(record: Any, index: int) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/image/prompt/mask")
    for key in ("id", "image", "prompt", "mask"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    try:
        image = validate_image(_open(record["image"], f"{where}.image"))
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{where}: {exc}") from exc
    if not isinstance(record["prompt"], str):
        raise ValueError(f"{where}: prompt must be a str")
    try:
        prompt = format_prompts([record["prompt"]])[0]
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{where}: {exc}") from exc
    mask = coerce_mask(record["mask"], image.size, f"{where}.mask")
    item = {"id": rid, "image": image, "prompt": prompt, "mask": mask}
    for key in ("class_id", "source_id", "source_row_group"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a labelled-mask dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, prompt, mask} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked, ids = [], set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    areas = [float(r["mask"].mean()) for r in checked]
    widths = [r["image"].width for r in checked]
    heights = [r["image"].height for r in checked]
    prompts = sorted({r["prompt"] for r in checked})
    return {
        "records": checked,
        "n_records": len(checked),
        "n_prompts": len(prompts),
        "prompts": prompts,
        "mask_area_fraction": {"min": min(areas), "max": max(areas), "mean": sum(areas) / len(areas)},
        "image_width": {"min": min(widths), "max": max(widths)},
        "image_height": {"min": min(heights), "max": max(heights)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size-prefixed) — the identity a split is made disjoint on."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())


def mask_digest(mask: np.ndarray) -> str:
    array = np.asarray(mask, dtype=bool)
    return _sha256_bytes(f"{array.shape[1]}x{array.shape[0]}:".encode() + np.packbits(array).tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over (id, image digest, prompt, mask digest)."""
    parts = sorted(f"{r['id']}:{image_digest(r['image'])}:{r['prompt']}:{mask_digest(r['mask'])}" for r in records)
    return _sha256_bytes("\n".join(parts).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating images."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n = len(unique)
    n_test = max(1, round(n * test_fraction))
    n_val = round(n * val_fraction)
    if n - n_test - n_val < 1:
        raise ValueError(f"{n} distinct images are too few to split into train/validation/test")
    return {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Records from a directory or zip holding images, mask images and a `masks.csv` with the columns `file`,
    `mask` and `prompt` (and optionally `id`); every listed file must exist and every image file must be listed
    (mask files are those a `mask` column names)."""
    source = Path(path)
    members: dict[str, bytes] = {}
    if source.is_dir():
        for file in sorted(source.rglob("*")):
            if file.is_file():
                members[file.name] = file.read_bytes()
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as archive:
            for info in archive.infolist():
                if not info.is_dir():
                    members[Path(info.filename).name] = archive.read(info)  # flattened; no extractall
    else:
        raise ValueError(f"{source} is neither a directory nor a zip file")
    if "masks.csv" not in members:
        raise ValueError("BYOD data must include masks.csv with the columns file, mask and prompt")
    rows = list(csv.DictReader(io.StringIO(members["masks.csv"].decode("utf-8-sig"))))
    if not rows or any(column not in rows[0] for column in ("file", "mask", "prompt")):
        raise ValueError("masks.csv must have the columns file, mask and prompt")
    out = []
    for row in rows:
        name = Path(str(row.get("file", "")).strip()).name
        mask_name = Path(str(row.get("mask", "")).strip()).name
        for needed in (name, mask_name):
            if needed not in members:
                raise ValueError(f"masks.csv names a missing file: {needed}")
        try:
            image = Image.open(io.BytesIO(members[name]))
            image.load()
            mask = Image.open(io.BytesIO(members[mask_name]))
            mask.load()
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"BYOD file is not a decodable image: {name} / {mask_name}") from exc
        rid = str(row.get("id", "") or "").strip()
        out.append({"id": rid or re.sub(r"[^A-Za-z0-9_.:-]", "_", Path(name).stem)[:64], "image": image.convert("RGB"), "prompt": str(row.get("prompt", "")), "mask": mask})
    listed = {Path(str(r.get(k, "")).strip()).name for r in rows for k in ("file", "mask")}
    unlisted = [n for n in members if n != "masks.csv" and n not in listed]
    if unlisted:
        raise ValueError(f"{len(unlisted)} file(s) have no masks.csv row, e.g. {unlisted[0]}")
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """A summary table (id, image size, prompt, mask area fraction, provenance) in the BYOD `masks.csv` column
    layout plus extras (`file`/`mask` name the id; the images themselves are not written)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "file", "mask", "prompt", "width", "height", "mask_area_fraction", "class_id", "source_id", "source_row_group"])
        for r in records:
            writer.writerow([r["id"], f"{r['id']}.jpg", f"{r['id']}_mask.png", r["prompt"], r["image"].width, r["image"].height, round(float(np.asarray(r["mask"], dtype=bool).mean()), 4), r.get("class_id", ""), r.get("source_id", ""), r.get("source_row_group", "")])
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `999e0328d9e1…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `ClipSegSegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "clipseg-rd64-refined",
  "modelId": "CIDAS/clipseg-rd64-refined",
  "revision": "999e0328d9e10b484360c477313983f9afdd7050",
  "files": [
    {
      "path": "README.md",
      "bytes": 596,
      "sha256": "c1e26ac022542015a04d804460bdc1159d753c4349b628e086f127475d2fcf25"
    },
    {
      "path": "config.json",
      "bytes": 4732,
      "sha256": "c023375966d31b3b1392764f7bd91df47098ce19f62f11b0263d8eedcf708bcd"
    },
    {
      "path": "merges.txt",
      "bytes": 524619,
      "sha256": "9fd691f7c8039210e0fced15865466c65820d09b63988b0174bfe25de299051a"
    },
    {
      "path": "model.safetensors",
      "bytes": 603049496,
      "sha256": "d00ca85d6b859f9d07b7cfb8ef26fe9771cb275b34c9368f2ecf603139307f55"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 380,
      "sha256": "4fb09ebcfd7651205ca8299b993c30088e5535ef350a72d46c5c4580eeac0440"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 472,
      "sha256": "c4864a9376a8401918425bed71fc14fc0e81f9b59ec45c1cf96cccb2df508eac"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 974,
      "sha256": "4c75d57fd9bd0be8478ad2d6f8b9cebdd4a45338eb108547329c2b6333476ca6"
    },
    {
      "path": "vocab.json",
      "bytes": 1059962,
      "sha256": "e089ad92ba36837a0d31433e555c8f45fe601ab5c221d4f607ded32d9f7a4349"
    }
  ],
  "totalBytes": 604641231
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = ClipSegSegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. FoodSeg103 records, the phrases and the split

`fetch_corpus` returns the eight pinned row groups from the cache under `weights/foodseg103/` or the Hub at the pinned parquet-conversion revision — `pyarrow` reads the shard's footer and exactly those row groups over HTTPS range requests; every cached file is re-hashed and every fetched row group refused on any SHA-256 or byte-total mismatch — and `read_corpus` turns each row into a record: the photograph, the ingredient class that covers the most pixels as the phrase (`largest_class`; background and *other ingredients* never qualify) and that class's pixels as the mask. `build_sample_dataset` draws a seeded image-level split (600 / 60 / 140). `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no image (by decoded-pixel digest) is shared, and the training split's summary table is written to `outputs/clipseg_segmentation_train.csv`.

Look for: 800 records over some sixty phrases with masks covering about a quarter of their images, three digests, and four refusal probes — a duplicate id, an empty mask, a mask of the wrong shape, and a dataset too small to use — each rejected before the model does anything.

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_zip = Path('work') / 'byod.zip'
    byod_zip.parent.mkdir(parents=True, exist_ok=True)
    byod_zip.write_bytes(payload)
    records = load_byod_dataset(byod_zip)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    t0 = time.perf_counter()
    corpus_groups = fetch_corpus(cache_dir='weights/foodseg103')
    corpus = read_corpus(corpus_groups)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} @ {CORPUS_REVISION[:12]} ({CORPUS_LICENSE})'
    raw_rows = {'row_groups': len(corpus_groups), 'images': sum(len(v) for v in corpus_groups.values()), 'records': len(corpus), 'bytes': sum(len(r['image']) + len(r['label']) for v in corpus_groups.values() for r in v), 'seconds': round(time.perf_counter() - t0, 1)}
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(train_records, 'outputs/clipseg_segmentation_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'prompts': manifest['n_prompts'], 'mask_area': {k: round(v, 3) for k, v in manifest['mask_area_fraction'].items()}, 'width': manifest['image_width'], 'height': manifest['image_height'], 'digest': manifest['digest'][:16] + '...'}})


def overlay(image, mask, colour=(220, 40, 40)):
    base = np.asarray(image.convert('RGB'), dtype=np.float32)
    out = base.copy()
    out[mask] = 0.45 * out[mask] + 0.55 * np.array(colour, dtype=np.float32)
    return Image.fromarray(out.round().astype(np.uint8))


example = train_records[0]
overlay(example['image'], example['mask']).save('outputs/clipseg_segmentation_example_record.png')
print({'example': {'id': example['id'], 'image': list(example['image'].size), 'prompt': example['prompt'], 'mask_area_fraction': round(float(example['mask'].mean()), 3)}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty mask': [{**train_records[0], 'mask': np.zeros_like(train_records[0]['mask'])}, *train_records[1:8]],
    'mask of the wrong shape': [{**train_records[0], 'mask': np.ones((8, 8), dtype=bool)}, *train_records[1:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Segment a drawn scene through the inference contract

The inference contract is exercised as the inference-only tutorial exercised it: a 640 × 480 scene drawn in code — a red circle, a blue square, a yellow triangle and a green ground band — with the exact masks the shapes were drawn from and two absent phrases (`a cat`, `the sky`) on purpose; a different image family from the food photographs, and a scene the adapted model will segment again in Section 9. `validate_inputs` applies exactly the checks `segment` applies (one image with sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE`, 1..`MAX_PROMPTS` distinct phrases of at most `MAX_PROMPT_CHARS` characters, a threshold in [0, 1]) and returns an input manifest; a duplicate-phrase request is validated too and its rejection recorded as a finding. `segment` returns one mask, probability map, area fraction, tight box and maximum probability per phrase — **the probabilities are an uncalibrated sigmoid** and the threshold is a **caller-owned request parameter**; an absent phrase still yields a map. `evaluation_report` with the drawn masks is `sample-sanity`: one `mask_iou` per drawn phrase and their mean, plumbing evidence for one drawing — a segmentation benchmark needs labelled masks, which Section 6 supplies. The inference-only card recorded a mean IoU of 0.86 on the four drawn shapes.

In [ ]:
def synthetic_scene(width=640, height=480):
    """Coloured shapes drawn with Pillow (no text); returns image + {phrase: boolean reference mask}."""
    image = Image.new('RGB', (width, height), (245, 245, 240))
    d = ImageDraw.Draw(image)
    shapes = [
        ('green grass', 'rectangle', [0, 320, 640, 480], (60, 179, 75)),
        ('a red circle', 'ellipse', [80, 80, 260, 260], (220, 40, 40)),
        ('a blue square', 'rectangle', [340, 90, 560, 300], (40, 70, 200)),
        ('a yellow triangle', 'polygon', [(200, 460), (320, 330), (440, 460)], (250, 200, 30)),
    ]
    masks = {}
    for phrase, kind, geometry, colour in shapes:
        getattr(d, kind)(geometry, fill=colour)
        reference = Image.new('1', image.size)
        getattr(ImageDraw.Draw(reference), kind)(geometry, fill=1)
        masks[phrase] = np.array(reference, dtype=bool)
    return image, masks


scene, scene_masks = synthetic_scene()
scene_prompts = list(scene_masks) + ['a cat', 'the sky']  # two absent phrases on purpose
scene_name = 'synthetic_shapes_640x480'
scene_sha256 = hashlib.sha256(np.asarray(scene).tobytes()).hexdigest()
THRESHOLD = MASK_THRESHOLD
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'LOGIT_SIZE': LOGIT_SIZE, 'MAX_PROMPTS': MAX_PROMPTS, 'MAX_PROMPT_CHARS': MAX_PROMPT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MASK_THRESHOLD': MASK_THRESHOLD, 'EXTRACT_LAYERS': list(EXTRACT_LAYERS), 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'EVAL_BATCH_SIZE': EVAL_BATCH_SIZE, 'device': pipe.device}})
input_manifest = validate_inputs(scene, scene_prompts, threshold=THRESHOLD, names=[scene_name])
try:
    validate_inputs(scene, ['a red circle', 'A red circle.'])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'duplicate-phrase-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/clipseg_segmentation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'scene': scene_name, 'sha256': scene_sha256[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})


def segment_scene(pipeline, label):
    started = time.perf_counter()
    result = pipeline.segment(scene, scene_prompts, threshold=THRESHOLD)
    seconds = round(time.perf_counter() - started, 3)
    checks = {
        'one_segment_per_phrase': [s['prompt'] for s in result['segments']] == result['queries'] and len(result['queries']) == len(scene_prompts),
        'probabilities_in_unit_interval': all(0.0 <= s['probability'].min() and s['probability'].max() <= 1.0 for s in result['segments']),
        'masks_at_input_resolution': all(s['mask'].shape == (scene.height, scene.width) for s in result['segments']),
        'identity_reported': result['model_id'] == MODEL_ID and result['model_revision'] == MODEL_REVISION,
    }
    if not all(checks.values()):
        raise RuntimeError(f'segment output failed a sanity check: {checks}')
    report = evaluation_report(result, scene_masks, sample_kind='synthetic (drawn in this notebook)')
    summary = {s['prompt']: {'area_fraction': round(s['area_fraction'], 3), 'max_probability': round(s['max_probability'], 3), 'bbox': s['bbox']} for s in result['segments']}
    ious = {m['reference']: round(m['value'], 3) for m in report['metrics'] if m['id'] == 'mask_iou'}
    miou = next((m['value'] for m in report['metrics'] if m['id'] == 'miou'), None)
    with open(f'outputs/clipseg_segmentation_scene_{label}.json', 'w', encoding='utf-8') as handle:
        json.dump({'segments': summary, 'report': report}, handle, indent=2, ensure_ascii=False)
    palette = [(220, 40, 40), (40, 70, 200), (250, 200, 30), (60, 179, 75), (160, 60, 200), (0, 170, 170)]
    sheet = np.asarray(scene, dtype=np.float32).copy()
    for index, s in enumerate(result['segments']):
        sheet[s['mask']] = 0.45 * sheet[s['mask']] + 0.55 * np.array(palette[index % len(palette)], dtype=np.float32)
    Image.fromarray(sheet.round().astype(np.uint8)).save(f'outputs/clipseg_segmentation_scene_{label}.png')
    print({label: {'seconds': seconds, 'checks': checks, 'mask_iou': ious, 'miou': None if miou is None else round(miou, 3), 'absent_phrases': {p: summary[p]['area_fraction'] for p in ('a cat', 'the sky')}, 'verdict': report['verdict']}})
    return summary, seconds, checks, report


frozen_scene, frozen_scene_seconds, frozen_scene_checks, frozen_scene_report = segment_scene(pipe, 'frozen')

## 6. Baselines and the frozen model on the test records

Two non-adapted baselines frame the adaptation, each scored by `segmentation_metrics` (carried in `metrics.py`): the **mean IoU** (the per-record intersection-over-union of the thresholded mask and the reference, averaged — the measure the epoch is selected on), the **micro IoU** (intersection over union pooled over every pixel of the split, so large masks weigh more), **Dice**, and **pixel precision** and **pixel recall** pooled over the split. The **empty-mask** baseline predicts no pixel and scores 0 by construction — the floor any segmenter must beat to do better than silence. The **full-mask** baseline predicts every pixel: its IoU is the reference's area fraction, what "the ingredient is somewhere in the photograph" buys without looking. The **frozen model** is scored by `pipe.evaluate`, which segments every record's phrase on its image in batches of `EVAL_BATCH_SIZE`, thresholds the sigmoid at `THRESHOLD` and scores the mask. Expect the frozen model **well above both baselines** — it is a phrase-conditioned segmenter and these are nameable things: the build record measured **@P:FROZEN_MIOU@** mean IoU on the 140 held-out records (@P:FROZEN_READ@); read four records' scores under their phrases.

In [ ]:
METRICS = ('miou', 'iou_micro', 'dice', 'pixel_precision', 'pixel_recall')

baseline_empty = empty_baseline(test_records)
baseline_full = full_baseline(test_records)
print({'empty_baseline': {k: round(baseline_empty[k], 3) for k in METRICS}, 'n': baseline_empty['n'], 'note': baseline_empty['baseline']})
print({'full_baseline': {k: round(baseline_full[k], 3) for k in METRICS}, 'note': baseline_full['baseline']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, threshold=THRESHOLD, batch_size=EVAL_BATCH_SIZE)
print({'frozen_model_test': {k: round(frozen_test[k], 3) for k in METRICS}, 'n': frozen_test['n'], 'predicted_area_fraction': round(frozen_test['predicted_area_fraction'], 3), 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
for row in frozen_test['rows'][:4]:
    print({'id': row['id'], 'prompt': row['prompt'], 'iou': round(row['iou'], 3), 'dice': round(row['dice'], 3), 'reference_pixels': row['reference'], 'predicted_pixels': row['predicted']})

## 7. Bounded fine-tuning of the decoder

`pipe.adapt` trains only the CLIPSeg decoder — the three transformer layers over the reduced activations, the FiLM conditioning, the reduce projections and the transposed convolution: 1,127,009 of 150,747,746 parameters — while the CLIP image and text towers stay frozen, exactly the split the upstream authors trained with. The loss is the **per-pixel binary cross-entropy** between the 352 × 352 decoder logits and the reference mask resampled to that grid — the upstream training objective. Because the towers are frozen, their outputs — the image activations at layers 3, 6 and 9 and the phrase embedding — are computed once per record under no gradient and cached in half precision on the host (the **frozen-tower cache**), and each step runs only the decoder on those cached activations: the logits equal the full model's exactly, at a fraction of the cost. AdamW without weight decay at a fixed learning rate, gradient clipping at 1.0, seeded shuffling, no scheduler, no augmentation. Epoch 0 records the frozen model's validation rates; every epoch is scored on the 60 validation records at `THRESHOLD`, and the epoch with the **highest validation mean IoU** (the earliest on ties) is kept.

Watch the validation mean IoU rise from @P:VAL_MIOU_0@ to @P:VAL_MIOU_BEST@ (epoch @P:BEST_EPOCH@ in the build record) while the loss drops from about @P:LOSS_1@ to @P:LOSS_LAST@: @P:ADAPTED_READ@.

In [ ]:
EPOCHS = 8  # @param {type:"integer"}
LEARNING_RATE = 3e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in METRICS})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, threshold=THRESHOLD, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'threshold': adapt_result['threshold'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'extract_layers': adapt_result['extract_layers'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'loss': adapt_result['loss'], 'cache_seconds': adapt_result['cache_seconds'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test records were never used for training or epoch selection, and no image appears in two splits. The adapted model is scored exactly as the frozen model was in Section 6 and the four systems are put side by side. Read it in this order: **mean IoU** first (the measure the epoch was selected on — the build record measured @P:FROZEN_MIOU@ → **@P:ADAPTED_MIOU@**), then **Dice** (@P:FROZEN_DICE@ → @P:ADAPTED_DICE@), then pixel **precision** and **recall** together (@P:FROZEN_PR@ → @P:ADAPTED_PR@ — a gain in one at the cost of the other is a moved threshold, not a better segmenter), then the predicted area against the reference area. The cell asserts the adapted mean IoU is at least the frozen one and above the full-mask baseline. One hundred and forty records from one seeded split give **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a result on one food dataset's largest-ingredient masks says nothing about other phrases, other images or your data until you measure them.

In [ ]:
adapted_test = pipe.evaluate(test_records, threshold=THRESHOLD, batch_size=EVAL_BATCH_SIZE)
adapted_val = pipe.evaluate(val_records, threshold=THRESHOLD, batch_size=EVAL_BATCH_SIZE)
comparison = {metric: {'empty': round(baseline_empty[metric], 3), 'full': round(baseline_full[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS}
comparison['area'] = {'reference_pixels': adapted_test['reference_pixels'], 'frozen_predicted_pixels': frozen_test['predicted_pixels'], 'adapted_predicted_pixels': adapted_test['predicted_pixels'], 'frozen_predicted_area_fraction': round(frozen_test['predicted_area_fraction'], 3), 'adapted_predicted_area_fraction': round(adapted_test['predicted_area_fraction'], 3)}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'threshold': THRESHOLD,
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'empty': {k: v for k, v in baseline_empty.items() if k != 'rows'}, 'full': {k: v for k, v in baseline_full.items() if k != 'rows'}},
    'frozen_test': {k: v for k, v in frozen_test.items() if k != 'rows'},
    'validation_metrics': {k: v for k, v in adapted_val.items() if k != 'rows'},
    'test_metrics': {k: v for k, v in adapted_test.items() if k != 'rows'},
    'per_record': [{**frozen_row, 'adapted_iou': adapted_row['iou'], 'adapted_dice': adapted_row['dice'], 'adapted_predicted': adapted_row['predicted']} for frozen_row, adapted_row in zip(frozen_test['rows'], adapted_test['rows'], strict=True)],
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/clipseg_segmentation_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['miou'] >= frozen_test['miou']
assert adapted_test['miou'] > baseline_full['miou']
print({'report': 'outputs/clipseg_segmentation_evaluation_report.json', 'adapted_beats_both_baselines': adapted_test['miou'] > max(baseline_empty['miou'], baseline_full['miou'])})

## 9. Look at the masks, segment the scene again, export the adapter and reload it

Six held-out records are written as panels (`outputs/clipseg_segmentation_examples/`: the photograph with the reference mask, the frozen mask and the adapted mask side by side, the phrase and both IoUs beneath) so the numbers can be checked by eye. The drawn scene from Section 5 is then segmented again by the adapted model — the decoder that was tuned serves every phrase, so this is a small look at what the adaptation did *outside* its phrase vocabulary and its corpus: the build record measured @P:DRAWING_AFTER@ — one drawing of evidence, not a measurement.

`pipe.save_artifact` writes the trained tensors — the decoder, about 4.5 MB in float32 — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the threshold the epoch was selected at, the training configuration and the epoch history (OUT8). `ClipSegSegmentationPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the decoder, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical masks on eight test records (VER4).

In [ ]:
import shutil

examples_dir = Path('outputs/clipseg_segmentation_examples')
shutil.rmtree(examples_dir, ignore_errors=True)
examples_dir.mkdir(parents=True)
caption_font = ImageFont.load_default(size=18)
adapted_items = pipe.segment_batch([(r['image'], r['prompt']) for r in test_records[:6]], threshold=THRESHOLD)
for record, frozen_row, adapted_row, adapted_item in zip(test_records[:6], frozen_test['rows'][:6], adapted_test['rows'][:6], adapted_items, strict=True):
    thumb = record['image'].convert('RGB')
    scale = 320 / max(thumb.size)
    size = (max(1, round(thumb.width * scale)), max(1, round(thumb.height * scale)))
    small = thumb.resize(size)
    ref_small = np.asarray(Image.fromarray(record['mask'].astype(np.uint8) * 255).resize(size, Image.NEAREST)) > 127
    ada_small = np.asarray(Image.fromarray(adapted_item['mask'].astype(np.uint8) * 255).resize(size, Image.NEAREST)) > 127
    panels = [overlay(small, ref_small, (60, 179, 75)), overlay(small, ada_small, (220, 40, 40))]
    sheet = Image.new('RGB', (sum(p.width for p in panels) + 8, panels[0].height + 56), (255, 255, 255))
    x = 0
    for panel in panels:
        sheet.paste(panel, (x, 0))
        x += panel.width + 8
    marker = ImageDraw.Draw(sheet)
    marker.text((8, panels[0].height + 6), f"{record['prompt']} — reference (green) | adapted (red); IoU frozen {frozen_row['iou']:.3f} -> adapted {adapted_row['iou']:.3f}", fill=(20, 20, 20), font=caption_font)
    sheet.save(examples_dir / f"{record['id']}.png")
print({'examples': sorted(p.name for p in examples_dir.iterdir()), 'panels': ['reference overlay', 'adapted overlay']})

adapted_scene, adapted_scene_seconds, adapted_scene_checks, adapted_scene_report = segment_scene(pipe, 'adapted')

artifact_dir = Path('outputs/clipseg_segmentation_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'clipseg_segmentation', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...', 'threshold': artifact_manifest['adapter']['threshold'], 'best_epoch': artifact_manifest['adapter']['best_epoch']})

reloaded = ClipSegSegmentationPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [item['mask'] for item in pipe.segment_batch([(r['image'], r['prompt']) for r in test_records[:8]], threshold=THRESHOLD)]
after = [item['mask'] for item in reloaded.segment_batch([(r['image'], r['prompt']) for r in test_records[:8]], threshold=THRESHOLD)]
parity = {'identical_masks': sum(bool(np.array_equal(a, b)) for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_masks'] == parity['of']

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'threshold': THRESHOLD,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'file': CORPUS_FILE, 'license': CORPUS_LICENSE, 'row_groups': sorted(ROW_GROUP_PINS), 'shard_bytes': CORPUS_BYTES, 'excluded_class_ids': list(EXCLUDED_CLASS_IDS)},
    'inference_contract': {'input_manifest': input_manifest, 'scene': {'name': scene_name, 'sha256': scene_sha256, 'prompts': scene_prompts}, 'frozen': {'segments': frozen_scene, 'seconds': frozen_scene_seconds, 'checks': frozen_scene_checks, 'report': frozen_scene_report}, 'adapted': {'segments': adapted_scene, 'seconds': adapted_scene_seconds, 'checks': adapted_scene_checks, 'report': adapted_scene_report}, 'output_files': ['outputs/clipseg_segmentation_scene_frozen.json', 'outputs/clipseg_segmentation_scene_adapted.json', 'outputs/clipseg_segmentation_scene_frozen.png', 'outputs/clipseg_segmentation_scene_adapted.png']},
    'comparison': comparison,
    'examples': 'outputs/clipseg_segmentation_examples',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pillow': PIL.__version__, 'device': pipe.device, 'dtype': 'float32'},
}
with open('outputs/clipseg_segmentation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

A phrase-conditioned segmenter trained on PhraseCut already finds the largest ingredient in a food photograph roughly when asked by name: the frozen model scores a mean IoU of @P:FROZEN_MIOU@ on the FoodSeg103 records. A bounded fine-tuning of its decoder on 600 records moves that to @P:ADAPTED_MIOU@ mean IoU and @P:ADAPTED_DICE@ Dice in the build record, with a 4.5 MB adapter that reloads mask-for-mask. That is the claim: the adaptation contract works end to end on a text-prompted segmenter with a real labelled set, and the numbers it produces are read as mean and micro IoU, Dice, precision and recall at one stated threshold, against two non-adapted baselines and the frozen model, with the predicted area beside them rather than in isolation. @P:SIBLING_COMPARISON@

The test split is 140 records from one seeded draw of one 800-record sample, the validation split that picks the epoch is 60, and every rate is at the one threshold `MASK_THRESHOLD` — not a benchmark, not a threshold sweep, not a measure of phrases the sample never asks (one phrase per image, the largest ingredient only). So a result here says the contract works on food photographs' largest ingredients, not that the adapted model handles other phrases, other image families or your masks. The decoder that was tuned serves every phrase: the drawn scene re-segmented in Section 9 is one drawing of evidence about what the tuning did outside its vocabulary (@P:DRAWING_AFTER@), not a measurement, and a deployment that segments other phrases must measure them after adapting. The towers were not adapted: what the image encoder cannot see stays unsegmented, and **the probabilities remain an uncalibrated sigmoid**.

Three things to carry to real data. **Baselines first:** the empty and full-mask rates on *your* masks, and the frozen model's predicted area, are the numbers to read before any adapted one. **Precision and recall together:** a gain in mean IoU that comes with a collapse of one of them is a moved threshold, and the threshold is yours to set on a validation split, not the test split. **Leakage:** keep every image in one split (the contract de-duplicates by decoded pixels) and split by source, session or scene when your images come from few sources.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled mask set, validate the demonstrated dataset contract without leakage, execute the inference contract for a drawn scene and a bounded fine-tuning of the decoder with the upstream objective, evaluate against two non-adapted baselines and the frozen model on an image-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, segmentation quality on any other phrase vocabulary or image family, calibration of the sigmoid, or production fitness.

**Optional experiments (they do not affect the default path):** raise `EPOCHS` and watch the validation mean IoU pick the epoch; change `LEARNING_RATE` by a factor of ten in either direction and read the curve; set `THRESHOLD` to `0.3` or `0.7` before Section 6 and read how precision and recall trade against each other for both the frozen and the adapted model; change `SPLIT_SEED` and read how much 140 records move; or bring your own masks through BYOD and read the two baselines before the adapted number.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/clipseg-rd64-refined/` and rerun Section 3. A `sha256` `ValueError` naming a parquet row group in Section 4: a cached `weights/foodseg103/validation-rg*.parquet` is incomplete — delete it and rerun Section 4.

## References

- Repository README: https://github.com/kurtvalcorza/clipseg-segmentation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/clipseg-segmentation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/clipseg-segmentation-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/CIDAS/clipseg-rd64-refined
- Upstream code: https://github.com/timojl/clipseg
- Image Segmentation Using Text and Image Prompts (Lüddecke and Ecker, CVPR 2022): https://arxiv.org/abs/2112.10003
- FoodSeg103 (Apache-2.0): https://huggingface.co/datasets/EduardoPacheco/FoodSeg103 — Wu, Fu, Liu, Lim, Hoi, Sun. A Large-Scale Benchmark for Food Image Segmentation (ACM MM 2021): https://arxiv.org/abs/2105.05409
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)